
# Kuroshio Daily R1 — Remaining 22 Days Interactive Manual Annotation
## VS Code + WSL + Jupyter (`ipympl`)

这个 Notebook 只用于处理 FIX6 中剩余的 22 个 `manual_review_required` 日期。

### 设计原则

- **当前日期的 50 m 速度场始终是人工判定主体**；
- 原 baseline failure 只作为红色参考线；
- 前一/后一已接受日期路径只用灰色虚线显示，帮助判断时间连续性，**不会自动成为真值**；
- 可选的“时间插值初始线”只用于减少点击量，最后必须由你根据当天速度场人工确认；
- 你点击的是粗控制点；脚本会在你选择的分支附近进行**局地速度脊细化**，不会跨到远处其它涡旋；
- 每一天保存：
  - 人工控制点；
  - 细化后的完整轴线；
  - 速度脊质量指标；
  - 人工确认记录；
  - QC PNG；
- 22 天全部完成后，脚本会生成：
  - `R1_FINAL_MANUAL22/tracks_final/`
  - `daily_dual_branch_paths_FINAL.npz`
  - 最终 R2 anchor readiness。

> 这些 22 天属于 **data curation / manual QC correction**，不能代替审稿人要求的 independent physical-axis validation。独立验证要在最终数据冻结后另做。



## 运行前

在 VS Code 的 WSL 终端中：

```bash
cd /mnt/d/KuroshioDailyR1
source ~/.venvs/kuroshio-r1/bin/activate
python -m pip install -U ipympl ipywidgets
```

然后确保 VS Code 选择的 Kernel 是你的 WSL Python 环境。

如果 `ipympl` 已安装，不需要重复安装。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, gc, shutil, warnings
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    CACHE50_DIR: str = "cache_50m"
    BASELINE_DIR: str = "tracks_baseline_FIX2_east0"
    FIX6_TRACK_DIR: str = "R1_FIX6/tracks_final"
    FIX6_REPORT_DIR: str = "R1_FIX6/reports"

    MANUAL_ROOT: str = "R1_MANUAL22"
    MANUAL_TRACK_DIR: str = "R1_MANUAL22/tracks_manual"
    MANUAL_META_DIR: str = "R1_MANUAL22/metadata"
    MANUAL_FIG_DIR: str = "R1_MANUAL22/figures"
    MANUAL_REPORT_DIR: str = "R1_MANUAL22/reports"

    FINAL_ROOT: str = "R1_FINAL_MANUAL22"
    FINAL_TRACK_DIR: str = "R1_FINAL_MANUAL22/tracks_final"
    FINAL_DATASET_DIR: str = "R1_FINAL_MANUAL22/dataset"
    FINAL_REPORT_DIR: str = "R1_FINAL_MANUAL22/reports"

    # display
    XMIN: float = 120.0
    XMAX: float = 148.0
    YMIN: float = 20.0
    YMAX: float = 42.0
    DISPLAY_STRIDE: int = 2
    QUIVER_STRIDE: int = 18

    # manual path refinement
    CONTROL_SNAP_RADIUS_KM: float = 18.0
    CONTROL_SNAP_DISTANCE_PENALTY: float = 0.35
    PATH_SAMPLE_SPACING_KM: float = 10.0

    RIDGE_HALF_KM: float = 24.0
    RIDGE_SAMPLES: int = 13
    RIDGE_DISTANCE_PENALTY: float = 0.22
    RIDGE_REFINE_PASSES: int = 2

    # save gate
    MIN_CONTROL_POINTS: int = 8
    MIN_MAX_LON: float = 145.8
    MIN_FINITE_SPEED_FRAC: float = 0.95

    # diagnostic warning only
    WARN_RIDGE_MEDIAN: float = 0.88
    WARN_RIDGE_P10: float = 0.70
    WARN_P95_TURN_DEG: float = 135.0

    # temporal initialization only
    TEMPORAL_INIT_CONTROL_POINTS: int = 22

cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)

CACHE50 = ROOT / cfg.CACHE50_DIR
BASE_DIR = ROOT / cfg.BASELINE_DIR
FIX6_TRACK = ROOT / cfg.FIX6_TRACK_DIR
FIX6_REPORT = ROOT / cfg.FIX6_REPORT_DIR

MANUAL_ROOT = ROOT / cfg.MANUAL_ROOT
MANUAL_TRACK = ROOT / cfg.MANUAL_TRACK_DIR
MANUAL_META = ROOT / cfg.MANUAL_META_DIR
MANUAL_FIG = ROOT / cfg.MANUAL_FIG_DIR
MANUAL_REPORT = ROOT / cfg.MANUAL_REPORT_DIR

FINAL_ROOT = ROOT / cfg.FINAL_ROOT
FINAL_TRACK = ROOT / cfg.FINAL_TRACK_DIR
FINAL_DATASET = ROOT / cfg.FINAL_DATASET_DIR
FINAL_REPORT = ROOT / cfg.FINAL_REPORT_DIR

for p in [
    MANUAL_ROOT, MANUAL_TRACK, MANUAL_META, MANUAL_FIG, MANUAL_REPORT,
    FINAL_ROOT, FINAL_TRACK, FINAL_DATASET, FINAL_REPORT
]:
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

# ===== 1. 交互环境 =====
import importlib.util

missing = [
    p for p in ["ipympl", "ipywidgets"]
    if importlib.util.find_spec(p) is None
]
if missing:
    raise RuntimeError(
        "缺少交互依赖: " + ", ".join(missing) + "\n"
        "请在WSL终端运行:\n"
        "python -m pip install -U ipympl ipywidgets\n"
        "然后重启Jupyter Kernel。"
    )

get_ipython().run_line_magic("matplotlib", "widget")

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.interpolate import RegularGridInterpolator, PchipInterpolator

plt.ioff()
print("interactive backend:", plt.get_backend())


In [ ]:

# ===== 2. 自动读取22个待人工处理日期 =====
manual_required_path = FIX6_REPORT / "manual_review_required_FIX6.csv"
final_report_path = FIX6_REPORT / "final_tracker_report_FIX6.csv"
recovery_summary_path = FIX6_REPORT / "recovery_summary_FIX6.csv"

for p in [manual_required_path, final_report_path]:
    if not p.exists():
        raise FileNotFoundError(p)

manual_required = pd.read_csv(
    manual_required_path,
    parse_dates=["date"]
).sort_values("date")

fix6_final_report = pd.read_csv(
    final_report_path,
    parse_dates=["date"]
).sort_values("date")

recovery_summary = (
    pd.read_csv(recovery_summary_path, parse_dates=["date"])
    if recovery_summary_path.exists()
    else pd.DataFrame()
)

UNRESOLVED_DATES = [
    pd.Timestamp(d).normalize()
    for d in manual_required["date"].tolist()
]

print("待人工标注日期数:", len(UNRESOLVED_DATES))
display(manual_required)

if len(UNRESOLVED_DATES) != 22:
    warnings.warn(
        f"当前读到 {len(UNRESOLVED_DATES)} 天，不是预期22天。"
        "Notebook仍会按实际列表工作。"
    )


In [ ]:

# ===== 3. 几何、路径插值、速度脊细化 =====
EARTH_R_KM = 6371.0088

def wrap_pi(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.asarray, (lon1,lat1,lon2,lat2))
    p1, p2 = np.deg2rad(lat1), np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    dp = p2-p1
    aa = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0, np.sqrt(aa)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2 = np.deg2rad(lat1),np.deg2rad(lat2)
    dl = np.deg2rad(lon2-lon1)
    yy = np.sin(dl)*np.cos(p2)
    xx = np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav = np.arctan2(yy,xx)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav = np.pi/2-east_angle
    phi1,lam1 = np.deg2rad(lat),np.deg2rad(lon)
    d = distance_km/EARTH_R_KM
    phi2 = np.arcsin(
        np.sin(phi1)*np.cos(d) +
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2 = lam1 + np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)), float(np.rad2deg(phi2))

def load_field(dt):
    p = CACHE50 / f"{pd.Timestamp(dt):%Y%m%d}.npz"
    if not p.exists():
        raise FileNotFoundError(p)
    with np.load(p) as z:
        return {
            "lon": np.asarray(z["lon"], dtype=np.float64),
            "lat": np.asarray(z["lat"], dtype=np.float64),
            "u": np.asarray(z["u"], dtype=np.float32),
            "v": np.asarray(z["v"], dtype=np.float32),
        }

def load_track(path):
    if path is None or not Path(path).exists():
        return None
    with np.load(path, allow_pickle=False) as z:
        return {
            "lon": np.asarray(z["lon"], dtype=np.float64),
            "lat": np.asarray(z["lat"], dtype=np.float64),
            "direction": np.asarray(z["direction"], dtype=np.float64)
                if "direction" in z.files else np.array([], dtype=float),
        }

def make_speed_interp(field):
    speed = np.hypot(field["u"], field["v"]).astype(np.float32)
    itp = RegularGridInterpolator(
        (field["lat"], field["lon"]),
        speed,
        bounds_error=False,
        fill_value=np.nan
    )
    return speed, itp

def interp_speed(itp, lon, lat):
    pts = np.column_stack([
        np.asarray(lat).ravel(),
        np.asarray(lon).ravel()
    ])
    return itp(pts).reshape(np.asarray(lon).shape)

def resample_polyline(lon, lat, spacing_km=None, n=None):
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    good = np.isfinite(lon) & np.isfinite(lat)
    lon, lat = lon[good], lat[good]

    if len(lon) < 2:
        return lon, lat

    seg = haversine_km(lon[:-1],lat[:-1],lon[1:],lat[1:])
    s = np.r_[0.0, np.cumsum(seg)]

    keep = np.r_[True, np.diff(s) > 1e-6]
    lon, lat, s = lon[keep], lat[keep], s[keep]

    if len(lon) < 2 or s[-1] <= 0:
        return lon, lat

    if n is None:
        spacing_km = cfg.PATH_SAMPLE_SPACING_KM if spacing_km is None else spacing_km
        n = max(2, int(np.ceil(s[-1]/spacing_km))+1)

    ss = np.linspace(0, s[-1], n)
    return np.interp(ss,s,lon), np.interp(ss,s,lat)

def temporal_neighbor(dt, direction):
    dt = pd.Timestamp(dt).normalize()
    good = fix6_final_report[fix6_final_report["completed"] == True].copy()

    if direction == "prev":
        good = good[good["date"] < dt].sort_values("date")
        if len(good)==0: return None,None
        d = pd.Timestamp(good.iloc[-1]["date"])
    else:
        good = good[good["date"] > dt].sort_values("date")
        if len(good)==0: return None,None
        d = pd.Timestamp(good.iloc[0]["date"])

    p = FIX6_TRACK / f"{d:%Y%m%d}.npz"
    return (d, load_track(p)) if p.exists() else (None,None)

def temporal_initial_path(dt, n=220):
    pd_prev, prev = temporal_neighbor(dt, "prev")
    pd_next, nxt = temporal_neighbor(dt, "next")

    if prev is None and nxt is None:
        return None

    if prev is not None:
        p_lon,p_lat = resample_polyline(prev["lon"],prev["lat"],n=n)
    if nxt is not None:
        n_lon,n_lat = resample_polyline(nxt["lon"],nxt["lat"],n=n)

    if prev is not None and nxt is not None:
        den = max(1, (pd_next-pd_prev).days)
        alpha = np.clip((pd.Timestamp(dt)-pd_prev).days/den, 0, 1)
        lon = (1-alpha)*p_lon + alpha*n_lon
        lat = (1-alpha)*p_lat + alpha*n_lat
        return lon,lat,pd_prev,pd_next

    if prev is not None:
        return p_lon,p_lat,pd_prev,None
    return n_lon,n_lat,None,pd_next

def snap_control_point(lon0,lat0,field,speed_itp):
    # 在用户点击附近的本地网格搜索，速度高但不能离点击太远
    lat_scale = 111.32
    lon_scale = 111.32*np.cos(np.deg2rad(lat0))

    dl = cfg.CONTROL_SNAP_RADIUS_KM/max(lon_scale,1e-6)
    da = cfg.CONTROL_SNAP_RADIUS_KM/lat_scale

    ix = np.where((field["lon"]>=lon0-dl)&(field["lon"]<=lon0+dl))[0]
    iy = np.where((field["lat"]>=lat0-da)&(field["lat"]<=lat0+da))[0]

    if len(ix)==0 or len(iy)==0:
        return float(lon0),float(lat0)

    LLON,LLAT = np.meshgrid(field["lon"][ix],field["lat"][iy])
    sp = np.hypot(
        field["u"][np.ix_(iy,ix)],
        field["v"][np.ix_(iy,ix)]
    ).astype(float)

    dist = haversine_km(lon0,lat0,LLON,LLAT)
    mask = np.isfinite(sp) & (dist<=cfg.CONTROL_SNAP_RADIUS_KM)

    if not mask.any():
        return float(lon0),float(lat0)

    smax = np.nanmax(sp[mask])
    sn = sp/(smax+1e-8)
    score = sn - cfg.CONTROL_SNAP_DISTANCE_PENALTY*(dist/cfg.CONTROL_SNAP_RADIUS_KM)
    score[~mask] = -np.inf

    j,i = np.unravel_index(np.nanargmax(score),score.shape)
    return float(LLON[j,i]),float(LLAT[j,i])

def smooth_control_polyline(points):
    pts = np.asarray(points,dtype=float)
    if len(pts)<2:
        return pts[:,0],pts[:,1]

    lon,lat = pts[:,0],pts[:,1]
    seg = haversine_km(lon[:-1],lat[:-1],lon[1:],lat[1:])
    s = np.r_[0.0,np.cumsum(seg)]
    keep = np.r_[True,np.diff(s)>1e-6]
    lon,lat,s = lon[keep],lat[keep],s[keep]

    if len(lon)<2:
        return lon,lat

    n = max(60,int(np.ceil(s[-1]/cfg.PATH_SAMPLE_SPACING_KM))+1)
    ss = np.linspace(0,s[-1],n)

    if len(lon) >= 3:
        fx = PchipInterpolator(s,lon)
        fy = PchipInterpolator(s,lat)
        return fx(ss),fy(ss)

    return np.interp(ss,s,lon),np.interp(ss,s,lat)

def ridge_refine(lon,lat,speed_itp):
    lon = np.asarray(lon,dtype=float).copy()
    lat = np.asarray(lat,dtype=float).copy()

    if len(lon)<5:
        return lon,lat

    offsets = np.linspace(
        -cfg.RIDGE_HALF_KM,
        cfg.RIDGE_HALF_KM,
        cfg.RIDGE_SAMPLES
    )

    for _ in range(cfg.RIDGE_REFINE_PASSES):
        new_lon,new_lat = lon.copy(),lat.copy()

        for i in range(1,len(lon)-1):
            tangent = angle_east_rad(
                lon[i-1],lat[i-1],
                lon[i+1],lat[i+1]
            )

            xs=[]; ys=[]
            for off in offsets:
                if abs(off)<1e-9:
                    x,y=lon[i],lat[i]
                else:
                    b = tangent + (np.pi/2 if off>0 else -np.pi/2)
                    x,y = forward_point(lon[i],lat[i],b,abs(float(off)))
                xs.append(x); ys.append(y)

            xs=np.asarray(xs); ys=np.asarray(ys)
            sp=interp_speed(speed_itp,xs,ys)
            good=np.isfinite(sp)

            if not good.any():
                continue

            smax=np.nanmax(sp[good])
            score = sp/(smax+1e-8) - cfg.RIDGE_DISTANCE_PENALTY*np.abs(offsets)/cfg.RIDGE_HALF_KM
            score[~good]=-np.inf
            k=int(np.nanargmax(score))

            new_lon[i]=xs[k]
            new_lat[i]=ys[k]

        lon,lat = new_lon,new_lat

    return lon,lat

def build_manual_path(control_points,field):
    if len(control_points) < 2:
        return None

    speed,speed_itp = make_speed_interp(field)

    snapped = [
        snap_control_point(x,y,field,speed_itp)
        for x,y in control_points
    ]

    lon,lat = smooth_control_polyline(snapped)
    lon,lat = ridge_refine(lon,lat,speed_itp)

    return {
        "lon":np.asarray(lon,dtype=np.float32),
        "lat":np.asarray(lat,dtype=np.float32),
        "control_points_raw":np.asarray(control_points,dtype=np.float32),
        "control_points_snapped":np.asarray(snapped,dtype=np.float32),
    }

def path_diagnostics(path,field):
    lon=np.asarray(path["lon"],float)
    lat=np.asarray(path["lat"],float)
    speed,speed_itp=make_speed_interp(field)

    sp=interp_speed(speed_itp,lon,lat)
    finite_frac=float(np.isfinite(sp).mean()) if len(sp) else 0.0

    # local ridge ratio
    rr=[]
    if len(lon)>=5:
        for i in range(2,len(lon)-2,2):
            tangent=angle_east_rad(
                lon[i-1],lat[i-1],
                lon[i+1],lat[i+1]
            )
            offsets=np.linspace(-35,35,15)
            xs=[];ys=[]
            for off in offsets:
                if abs(off)<1e-9:
                    x,y=lon[i],lat[i]
                else:
                    b=tangent+(np.pi/2 if off>0 else -np.pi/2)
                    x,y=forward_point(lon[i],lat[i],b,abs(float(off)))
                xs.append(x);ys.append(y)
            ss=interp_speed(speed_itp,np.asarray(xs),np.asarray(ys))
            if np.isfinite(ss).any() and np.isfinite(sp[i]):
                rr.append(float(sp[i]/(np.nanmax(ss)+1e-8)))

    rr=np.asarray(rr,float)

    path_km=float(np.nansum(haversine_km(lon[:-1],lat[:-1],lon[1:],lat[1:]))) if len(lon)>1 else 0.0

    if len(lon)>3:
        dirs=np.array([
            angle_east_rad(lon[i],lat[i],lon[i+1],lat[i+1])
            for i in range(len(lon)-1)
        ])
        turns=np.abs(np.rad2deg(wrap_pi(np.diff(dirs))))
        p95_turn=float(np.nanpercentile(turns,95))
    else:
        p95_turn=np.nan

    pd_prev,prev=temporal_neighbor(pd.NaT if False else CURRENT_DATE_PLACEHOLDER,"prev")
    # placeholder never called; temporal distances are appended outside

    return {
        "max_lon":float(np.nanmax(lon)) if len(lon) else np.nan,
        "final_lon":float(lon[-1]) if len(lon) else np.nan,
        "final_lat":float(lat[-1]) if len(lat) else np.nan,
        "path_km":path_km,
        "finite_speed_frac":finite_frac,
        "ridge_median":float(np.nanmedian(rr)) if len(rr) else np.nan,
        "ridge_p10":float(np.nanpercentile(rr,10)) if len(rr) else np.nan,
        "ridge_low_frac":float(np.mean(rr<0.75)) if len(rr) else np.nan,
        "p95_turn_deg":p95_turn,
    }

def symmetric_path_distance_km(path_a,path_b,sample_step=3):
    if path_a is None or path_b is None:
        return np.nan

    a_lon,a_lat=resample_polyline(path_a["lon"],path_a["lat"],n=180)
    b_lon,b_lat=resample_polyline(path_b["lon"],path_b["lat"],n=180)

    lat0=float(np.nanmean(np.r_[a_lat,b_lat]))
    lon0=float(np.nanmean(np.r_[a_lon,b_lon]))

    def xy(lon,lat):
        x=(lon-lon0)*111.32*np.cos(np.deg2rad(lat0))
        y=(lat-lat0)*111.32
        return np.column_stack([x,y])

    A=xy(a_lon[::sample_step],a_lat[::sample_step])
    B=xy(b_lon[::sample_step],b_lat[::sample_step])
    d=np.sqrt(((A[:,None,:]-B[None,:,:])**2).sum(axis=2))
    return float(np.median(np.r_[d.min(axis=1),d.min(axis=0)]))

# overwrite diagnostics wrapper without the placeholder
def full_diagnostics(dt,path,field):
    lon=np.asarray(path["lon"],float)
    lat=np.asarray(path["lat"],float)
    speed,speed_itp=make_speed_interp(field)
    sp=interp_speed(speed_itp,lon,lat)
    finite_frac=float(np.isfinite(sp).mean()) if len(sp) else 0.0

    rr=[]
    if len(lon)>=5:
        for i in range(2,len(lon)-2,2):
            tangent=angle_east_rad(lon[i-1],lat[i-1],lon[i+1],lat[i+1])
            offsets=np.linspace(-35,35,15)
            xs=[];ys=[]
            for off in offsets:
                if abs(off)<1e-9:
                    x,y=lon[i],lat[i]
                else:
                    b=tangent+(np.pi/2 if off>0 else -np.pi/2)
                    x,y=forward_point(lon[i],lat[i],b,abs(float(off)))
                xs.append(x);ys.append(y)
            ss=interp_speed(speed_itp,np.asarray(xs),np.asarray(ys))
            if np.isfinite(ss).any() and np.isfinite(sp[i]):
                rr.append(float(sp[i]/(np.nanmax(ss)+1e-8)))

    rr=np.asarray(rr,float)

    path_km=float(np.nansum(haversine_km(lon[:-1],lat[:-1],lon[1:],lat[1:]))) if len(lon)>1 else 0.0
    if len(lon)>3:
        dirs=np.array([
            angle_east_rad(lon[i],lat[i],lon[i+1],lat[i+1])
            for i in range(len(lon)-1)
        ])
        turns=np.abs(np.rad2deg(wrap_pi(np.diff(dirs))))
        p95_turn=float(np.nanpercentile(turns,95))
    else:
        p95_turn=np.nan

    _,prev=temporal_neighbor(dt,"prev")
    _,nxt=temporal_neighbor(dt,"next")
    pth={"lon":lon,"lat":lat}

    return {
        "max_lon":float(np.nanmax(lon)) if len(lon) else np.nan,
        "final_lon":float(lon[-1]) if len(lon) else np.nan,
        "final_lat":float(lat[-1]) if len(lat) else np.nan,
        "path_km":path_km,
        "finite_speed_frac":finite_frac,
        "ridge_median":float(np.nanmedian(rr)) if len(rr) else np.nan,
        "ridge_p10":float(np.nanpercentile(rr,10)) if len(rr) else np.nan,
        "ridge_low_frac":float(np.mean(rr<0.75)) if len(rr) else np.nan,
        "p95_turn_deg":p95_turn,
        "prev_axis_distance_km":symmetric_path_distance_km(pth,prev),
        "next_axis_distance_km":symmetric_path_distance_km(pth,nxt),
    }


In [ ]:

# ===== 4. 人工标注器 =====
class ManualAxisAnnotator:
    def __init__(self, dates):
        self.dates = [pd.Timestamp(d).normalize() for d in dates]
        self.index = 0
        self.dt = None
        self.field = None
        self.speed = None
        self.control_points = []
        self.path = None
        self.metrics = None

        self.fig, self.ax = plt.subplots(figsize=(11.5,7.5))
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)

        self.date_label = widgets.HTML()
        self.status = widgets.HTML()

        self.prev_btn = widgets.Button(description="◀ 上一天")
        self.next_btn = widgets.Button(description="下一天 ▶")
        self.auto_btn = widgets.Button(description="时间邻日初始化")
        self.refine_btn = widgets.Button(description="构建/细化路径", button_style="info")
        self.undo_btn = widgets.Button(description="撤销点")
        self.clear_btn = widgets.Button(description="清空")
        self.reload_btn = widgets.Button(description="载入已保存")
        self.save_btn = widgets.Button(description="保存并下一天", button_style="success")
        self.skip_btn = widgets.Button(description="仅下一天")
        self.quiver_box = widgets.Checkbox(value=False, description="显示流向箭头")
        self.confirm_box = widgets.Checkbox(value=False, description="我已确认路径沿当天黑潮主流轴")
        self.note = widgets.Textarea(
            placeholder="可选备注，例如：日本南侧强涡，人工沿东侧高速支路修正",
            description="备注",
            layout=widgets.Layout(width="720px",height="65px")
        )

        self.prev_btn.on_click(lambda b:self.move(-1))
        self.next_btn.on_click(lambda b:self.move(1))
        self.auto_btn.on_click(self.auto_init)
        self.refine_btn.on_click(self.refine)
        self.undo_btn.on_click(self.undo)
        self.clear_btn.on_click(self.clear)
        self.reload_btn.on_click(self.reload_saved)
        self.save_btn.on_click(self.save_and_next)
        self.skip_btn.on_click(lambda b:self.move(1))
        self.quiver_box.observe(lambda ch:self.redraw(), names="value")

        controls = widgets.VBox([
            self.date_label,
            widgets.HBox([
                self.prev_btn,self.next_btn,self.auto_btn,self.refine_btn,
                self.undo_btn,self.clear_btn,self.reload_btn
            ]),
            widgets.HBox([
                self.quiver_box,self.confirm_box,self.save_btn,self.skip_btn
            ]),
            self.note,
            self.status
        ])

        display(controls)
        display(self.fig.canvas)
        self.load_date(0)

    def _manual_track_path(self,dt):
        return MANUAL_TRACK / f"{pd.Timestamp(dt):%Y%m%d}.npz"

    def _manual_meta_path(self,dt):
        return MANUAL_META / f"{pd.Timestamp(dt):%Y%m%d}.json"

    def load_date(self,index):
        self.index = int(np.clip(index,0,len(self.dates)-1))
        self.dt = self.dates[self.index]
        self.field = load_field(self.dt)
        self.speed = np.hypot(self.field["u"],self.field["v"]).astype(np.float32)
        self.control_points = []
        self.path = None
        self.metrics = None
        self.confirm_box.value = False
        self.note.value = ""

        # if already saved, auto reload
        if self._manual_track_path(self.dt).exists():
            self.reload_saved(None)
        else:
            self.redraw()

        done = sum((MANUAL_TRACK/f"{d:%Y%m%d}.npz").exists() for d in self.dates)
        reason = manual_required.loc[
            manual_required["date"]==self.dt,"reason"
        ]
        reason = reason.iloc[0] if len(reason) else "unknown"

        self.date_label.value = (
            f"<h3>{self.index+1}/{len(self.dates)} — {self.dt.date()}</h3>"
            f"<b>FIX6原因:</b> {reason} &nbsp;&nbsp; "
            f"<b>已完成:</b> {done}/{len(self.dates)}"
        )
        self._set_status("左键按顺序添加控制点；右键删除最近控制点。建议从台湾东侧一路点击到 ≥146°E。")

    def move(self,delta):
        self.load_date(self.index+delta)

    def _set_status(self,msg,color="#333"):
        self.status.value = f"<div style='color:{color};font-size:14px'>{msg}</div>"

    def _baseline_track(self):
        p=BASE_DIR/f"{self.dt:%Y%m%d}.npz"
        return load_track(p)

    def _neighbors(self):
        return temporal_neighbor(self.dt,"prev"),temporal_neighbor(self.dt,"next")

    def redraw(self):
        self.ax.clear()
        st=max(1,cfg.DISPLAY_STRIDE)

        self.ax.imshow(
            self.speed[::st,::st],
            origin="lower",
            extent=[
                float(self.field["lon"][0]),float(self.field["lon"][-1]),
                float(self.field["lat"][0]),float(self.field["lat"][-1])
            ],
            aspect="auto",
            interpolation="nearest"
        )

        base=self._baseline_track()
        if base is not None:
            self.ax.plot(
                base["lon"],base["lat"],
                "--",lw=1.1,alpha=.9,label="FIX6 baseline failure"
            )

        (pd_prev,prev),(pd_next,nxt)=self._neighbors()
        if prev is not None:
            self.ax.plot(
                prev["lon"],prev["lat"],
                color="0.65",ls="--",lw=1.0,
                label=f"prev accepted {pd_prev.date()}"
            )
        if nxt is not None:
            self.ax.plot(
                nxt["lon"],nxt["lat"],
                color="0.45",ls=":",lw=1.0,
                label=f"next accepted {pd_next.date()}"
            )

        if self.quiver_box.value:
            qs=max(4,cfg.QUIVER_STRIDE)
            lonq=self.field["lon"][::qs]
            latq=self.field["lat"][::qs]
            uq=self.field["u"][::qs,::qs]
            vq=self.field["v"][::qs,::qs]
            self.ax.quiver(
                lonq,latq,uq,vq,
                color="white",alpha=.55,
                scale=25,width=.0018
            )

        if self.control_points:
            pts=np.asarray(self.control_points,float)
            self.ax.plot(
                pts[:,0],pts[:,1],
                "o-",ms=4,lw=.8,label="manual control points"
            )

        if self.path is not None:
            self.ax.plot(
                self.path["lon"],self.path["lat"],
                lw=2.2,label="manual refined axis"
            )
            self.ax.scatter(
                [self.path["lon"][-1]],[self.path["lat"][-1]],
                marker="x",s=42,label="manual end"
            )

        self.ax.set_xlim(cfg.XMIN,cfg.XMAX)
        self.ax.set_ylim(cfg.YMIN,cfg.YMAX)
        self.ax.set_xlabel("Longitude")
        self.ax.set_ylabel("Latitude")
        self.ax.set_title(
            f"{self.dt.date()} | 50 m speed | current-day field is the primary reference"
        )
        self.ax.legend(loc="best",fontsize=7)
        self.fig.canvas.draw_idle()

    def _on_click(self,event):
        if event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return

        if event.button == 1:
            self.control_points.append((float(event.xdata),float(event.ydata)))
            self.path=None
            self.metrics=None
            self.confirm_box.value=False
            self.redraw()
            self._set_status(
                f"已添加 {len(self.control_points)} 个控制点。"
                "点击“构建/细化路径”查看速度脊细化结果。"
            )

        elif event.button == 3 and self.control_points:
            x,y=float(event.xdata),float(event.ydata)
            dd=[
                float(haversine_km(px,py,x,y))
                for px,py in self.control_points
            ]
            k=int(np.argmin(dd))
            self.control_points.pop(k)
            self.path=None
            self.metrics=None
            self.confirm_box.value=False
            self.redraw()
            self._set_status(f"已删除最近控制点，当前 {len(self.control_points)} 个。")

    def undo(self,_):
        if self.control_points:
            self.control_points.pop()
            self.path=None
            self.metrics=None
            self.confirm_box.value=False
            self.redraw()

    def clear(self,_):
        self.control_points=[]
        self.path=None
        self.metrics=None
        self.confirm_box.value=False
        self.redraw()
        self._set_status("已清空。")

    def auto_init(self,_):
        init=temporal_initial_path(self.dt,n=220)
        if init is None:
            self._set_status("没有可用邻日路径。", "#b00")
            return

        lon,lat,pd_prev,pd_next=init
        ids=np.linspace(
            0,len(lon)-1,
            cfg.TEMPORAL_INIT_CONTROL_POINTS
        ).astype(int)

        self.control_points=[
            (float(lon[i]),float(lat[i]))
            for i in ids
        ]
        self.confirm_box.value=False
        self.refine(None)

        self._set_status(
            f"已用邻日路径生成初始控制线（prev={pd_prev}, next={pd_next}）。"
            "<b>请只把它当初始参考，必须根据当天速度场人工核对并修正。</b>",
            "#7a4b00"
        )

    def refine(self,_):
        if len(self.control_points)<cfg.MIN_CONTROL_POINTS:
            self._set_status(
                f"控制点不足：{len(self.control_points)}。建议至少 {cfg.MIN_CONTROL_POINTS} 个。",
                "#b00"
            )
            return

        self.path=build_manual_path(self.control_points,self.field)
        if self.path is None:
            self._set_status("无法构建路径。", "#b00")
            return

        self.metrics=full_diagnostics(self.dt,self.path,self.field)
        self.confirm_box.value=False
        self.redraw()

        m=self.metrics
        warn=[]
        if m["max_lon"]<cfg.MIN_MAX_LON:
            warn.append("未到146°E")
        if m["finite_speed_frac"]<cfg.MIN_FINITE_SPEED_FRAC:
            warn.append("存在较多无效流速点")
        if np.isfinite(m["ridge_median"]) and m["ridge_median"]<cfg.WARN_RIDGE_MEDIAN:
            warn.append("ridge median偏低")
        if np.isfinite(m["ridge_p10"]) and m["ridge_p10"]<cfg.WARN_RIDGE_P10:
            warn.append("ridge P10偏低")
        if np.isfinite(m["p95_turn_deg"]) and m["p95_turn_deg"]>cfg.WARN_P95_TURN_DEG:
            warn.append("转向过剧烈")

        msg=(
            f"<b>max_lon={m['max_lon']:.2f}°E</b> | "
            f"finite={m['finite_speed_frac']:.1%} | "
            f"ridge_med={m['ridge_median']:.3f} | "
            f"ridge_p10={m['ridge_p10']:.3f} | "
            f"P95_turn={m['p95_turn_deg']:.1f}° | "
            f"prev_dist={m['prev_axis_distance_km']:.1f} km | "
            f"next_dist={m['next_axis_distance_km']:.1f} km"
        )
        if warn:
            msg += "<br><b>警告:</b> " + "；".join(warn)
            self._set_status(msg,"#b05a00")
        else:
            self._set_status(msg,"#087f23")

    def reload_saved(self,_):
        p=self._manual_track_path(self.dt)
        meta=self._manual_meta_path(self.dt)
        if not p.exists():
            self._set_status("当前日期没有已保存人工轴。","#b00")
            return

        with np.load(p) as z:
            self.path={
                "lon":np.asarray(z["lon"],dtype=np.float32),
                "lat":np.asarray(z["lat"],dtype=np.float32),
                "control_points_raw":np.asarray(z["control_points_raw"],dtype=np.float32),
                "control_points_snapped":np.asarray(z["control_points_snapped"],dtype=np.float32),
            }
        self.control_points=[
            tuple(map(float,p))
            for p in self.path["control_points_raw"]
        ]
        self.metrics=full_diagnostics(self.dt,self.path,self.field)
        if meta.exists():
            md=json.loads(meta.read_text(encoding="utf-8"))
            self.note.value=md.get("note","")
        self.confirm_box.value=False
        self.redraw()
        self._set_status("已载入该日已保存人工轴。","#087f23")

    def save_and_next(self,_):
        if self.path is None or self.metrics is None:
            self._set_status("请先构建/细化路径。","#b00")
            return

        if not self.confirm_box.value:
            self._set_status("保存前请勾选“我已确认路径沿当天黑潮主流轴”。","#b00")
            return

        m=self.metrics
        hard_fail=[]
        if len(self.control_points)<cfg.MIN_CONTROL_POINTS:
            hard_fail.append("控制点过少")
        if m["max_lon"]<cfg.MIN_MAX_LON:
            hard_fail.append(f"max_lon={m['max_lon']:.2f}<145.8")
        if m["finite_speed_frac"]<cfg.MIN_FINITE_SPEED_FRAC:
            hard_fail.append(f"finite={m['finite_speed_frac']:.1%}")

        if hard_fail:
            self._set_status(
                "不能保存："+"；".join(hard_fail),
                "#b00"
            )
            return

        out=self._manual_track_path(self.dt)
        np.savez_compressed(
            out,
            lon=self.path["lon"],
            lat=self.path["lat"],
            direction=np.array([],dtype=np.float32),
            completed=np.array(True),
            loop_events=np.array(-1),
            n_points=np.array(len(self.path["lon"])),
            stop_reason=np.array("manual_qc_corrected"),
            tracker_version=np.array("R1_MANUAL22_V1"),
            control_points_raw=self.path["control_points_raw"],
            control_points_snapped=self.path["control_points_snapped"],
        )

        (pd_prev,_),(pd_next,_)=self._neighbors()
        metadata={
            "date":str(self.dt.date()),
            "source":"manual_qc_corrected",
            "note":self.note.value,
            "previous_reference_date":str(pd_prev.date()) if pd_prev is not None else None,
            "next_reference_date":str(pd_next.date()) if pd_next is not None else None,
            "metrics":{k:(None if not np.isfinite(v) else float(v)) for k,v in m.items()},
            "config":{
                "control_snap_radius_km":cfg.CONTROL_SNAP_RADIUS_KM,
                "ridge_half_km":cfg.RIDGE_HALF_KM,
                "ridge_refine_passes":cfg.RIDGE_REFINE_PASSES,
                "path_sample_spacing_km":cfg.PATH_SAMPLE_SPACING_KM,
            },
            "n_raw_control_points":len(self.control_points),
        }
        self._manual_meta_path(self.dt).write_text(
            json.dumps(metadata,indent=2,ensure_ascii=False),
            encoding="utf-8"
        )

        # 当前只有一张活动图，保存不会形成批量绘图内存峰值
        self.fig.savefig(
            MANUAL_FIG/f"{self.dt:%Y%m%d}_manual_qc.png",
            dpi=110,
            bbox_inches="tight"
        )

        self._set_status("已保存。","#087f23")

        if self.index < len(self.dates)-1:
            self.load_date(self.index+1)

annotator = ManualAxisAnnotator(UNRESOLVED_DATES)



## 标注操作建议

对每一天：

1. 先观察当天速度场最亮的连续高速带；
2. 可点“时间邻日初始化”快速得到一个初始线；
3. **不要直接相信初始化结果**，特别检查：
   - 台湾以东；
   - 九州南侧 / 吐噶喇；
   - 日本南侧大蛇行；
   - 130–140°E 多涡区域；
   - KE 向东延伸；
4. 如果初始线走错：
   - 右键删除附近控制点；
   - 左键重新沿正确高速带补点；
   - 点“构建/细化路径”；
5. 路径至少延伸到 145.8–146°E；
6. 确认绿色/新路径沿当天黑潮主流高速带后，勾选确认并保存。

### 控制点密度

正常段可以稀一些；弯曲、多涡和分叉区域要密一些。

大多数日期建议约 **15–30 个控制点**。


In [ ]:

# ===== 5. 人工标注完成度审计 =====
rows=[]
for dt in UNRESOLVED_DATES:
    p=MANUAL_TRACK/f"{dt:%Y%m%d}.npz"
    meta=MANUAL_META/f"{dt:%Y%m%d}.json"

    row={
        "date":dt,
        "saved":p.exists(),
        "metadata":meta.exists(),
    }

    if p.exists():
        with np.load(p) as z:
            row["n_points"]=len(z["lon"])
            row["max_lon"]=float(np.nanmax(z["lon"]))
            row["n_control_points"]=len(z["control_points_raw"])
    rows.append(row)

manual_status=pd.DataFrame(rows)
manual_status.to_csv(
    MANUAL_REPORT/"manual_annotation_status.csv",
    index=False,
    encoding="utf-8-sig"
)

display(manual_status)
print("完成:",int(manual_status.saved.sum()),"/",len(manual_status))
print("未完成:",manual_status.loc[~manual_status.saved,"date"].dt.strftime("%Y-%m-%d").tolist())


In [ ]:

# ===== 6. 22天全部完成后，冻结最终570天轨迹 =====
# 只有所有manual日期都完成才允许执行。
manual_status = pd.read_csv(
    MANUAL_REPORT/"manual_annotation_status.csv",
    parse_dates=["date"]
)

missing = manual_status.loc[
    ~manual_status["saved"].astype(bool),
    "date"
].tolist()

if missing:
    raise RuntimeError(
        "还有未完成人工标注日期:\n" +
        "\n".join(str(pd.Timestamp(d).date()) for d in missing)
    )

# 清空/重建FINAL_TRACK，避免残留旧文件
if FINAL_TRACK.exists():
    for p in FINAL_TRACK.glob("*.npz"):
        p.unlink()
FINAL_TRACK.mkdir(parents=True,exist_ok=True)

# 先复制FIX6已有548条
for p in FIX6_TRACK.glob("*.npz"):
    shutil.copy2(p, FINAL_TRACK/p.name)

# 再用22条人工轴覆盖/补齐
for dt in UNRESOLVED_DATES:
    src=MANUAL_TRACK/f"{dt:%Y%m%d}.npz"
    dst=FINAL_TRACK/src.name
    shutil.copy2(src,dst)

all_final_tracks=sorted(FINAL_TRACK.glob("*.npz"))
print("最终track数量:",len(all_final_tracks))

if len(all_final_tracks) < 570:
    warnings.warn(
        f"当前只有 {len(all_final_tracks)} 个最终track。"
        "请核对FIX6原始可计算日期数。"
    )


In [ ]:

# ===== 7. 生成 FINAL 双分支日尺度数据集 =====
from scipy.interpolate import interp1d

TARGET_LATS_1=np.arange(22.0,25.5+1e-9,1/12)
TARGET_LONS_2=np.arange(122.0,145.0+1e-9,1/12)

def branch1_from_track(lons,lats):
    valid=np.isfinite(lons)&np.isfinite(lats)&(lats<=25.7)
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]
    if len(lats)<2:
        return None

    df=(
        pd.DataFrame({"lat":lats,"lon":lons})
        .groupby("lat",as_index=False)
        .lon.mean()
        .sort_values("lat")
    )
    if len(df)<2:
        return None

    f=interp1d(
        df.lat.values,df.lon.values,
        bounds_error=False,
        fill_value="extrapolate"
    )
    return f(TARGET_LATS_1).astype(np.float32)

def branch2_from_track(lons,lats):
    valid=np.isfinite(lons)&np.isfinite(lats)
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]
    if len(lons)<2:
        return None

    vals=[]
    for lo in np.sort(np.unique(lons)):
        yy=lats[lons==lo]
        vals.append((
            lo,
            np.max(yy) if lo<=125 else np.mean(yy)
        ))

    arr=np.asarray(vals,float)
    if len(arr)<2:
        return None

    f=interp1d(
        arr[:,0],arr[:,1],
        bounds_error=False,
        fill_value="extrapolate"
    )
    return f(TARGET_LONS_2).astype(np.float32)

# source map from FIX6
src_map={}
for r in fix6_final_report.itertuples(index=False):
    if bool(r.completed):
        src_map[pd.Timestamp(r.date).normalize()]=str(r.source)

for dt in UNRESOLVED_DATES:
    src_map[dt]="manual_qc_corrected"

D=[];B1=[];B2=[];SRC=[]

for p in sorted(FINAL_TRACK.glob("*.npz")):
    dt=pd.Timestamp(p.stem)

    with np.load(p) as z:
        lon=np.asarray(z["lon"],float)
        lat=np.asarray(z["lat"],float)

    b1=branch1_from_track(lon,lat)
    b2=branch2_from_track(lon,lat)

    if (
        b1 is not None and b2 is not None and
        np.isfinite(b1).all() and np.isfinite(b2).all()
    ):
        D.append(dt)
        B1.append(b1)
        B2.append(b2)
        SRC.append(src_map.get(dt,"unknown"))

B1=np.stack(B1)
B2=np.stack(B2)

dataset_path=FINAL_DATASET/"daily_dual_branch_paths_FINAL.npz"
np.savez_compressed(
    dataset_path,
    dates=np.array([d.strftime("%Y-%m-%d") for d in D]),
    branch1_lon=B1,
    branch1_target_lats=TARGET_LATS_1,
    branch2_lat=B2,
    branch2_target_lons=TARGET_LONS_2,
    source=np.asarray(SRC)
)

meta=pd.DataFrame({
    "date":D,
    "source":SRC
})
meta.to_csv(
    FINAL_DATASET/"daily_dual_branch_meta_FINAL.csv",
    index=False,
    encoding="utf-8-sig"
)

print("FINAL dataset:",dataset_path)
print("FINAL dual-branch days:",len(D))
display(meta.source.value_counts().to_frame("count"))


In [ ]:

# ===== 8. 最终 R2 Anchor Readiness =====
R2_LOOKBACKS=(180,120,90,60,30)
R2_LEADS=(30,60,90)
MAX_INPUT_GAP=2
MAX_MISSING_FRAC=.03

avail=pd.Series(
    False,
    index=pd.date_range(min(D),max(D),freq="D")
)
avail.loc[pd.DatetimeIndex(D)]=True

def longest_missing_run(mask):
    best=cur=0
    for v in mask:
        if bool(v):
            cur+=1
            best=max(best,cur)
        else:
            cur=0
    return best

rows=[]

for lb in R2_LOOKBACKS:
    total=0
    min_issue=avail.index.min()+pd.Timedelta(days=lb-1)
    max_issue=avail.index.max()-pd.Timedelta(days=max(R2_LEADS))

    if min_issue<=max_issue:
        for issue in pd.date_range(min_issue,max_issue,freq="D"):
            win=avail.loc[
                issue-pd.Timedelta(days=lb-1):issue
            ]
            missing=~win.to_numpy()

            if missing.mean()>MAX_MISSING_FRAC:
                continue
            if longest_missing_run(missing)>MAX_INPUT_GAP:
                continue

            targets=[
                issue+pd.Timedelta(days=x)
                for x in R2_LEADS
            ]

            if any(
                t not in avail.index or not bool(avail.loc[t])
                for t in targets
            ):
                continue

            total+=1

    rows.append({
        "lookback_days":lb,
        "anchors_total":total
    })

readiness=pd.DataFrame(rows)
readiness.to_csv(
    FINAL_REPORT/"R2_anchor_readiness_FINAL.csv",
    index=False,
    encoding="utf-8-sig"
)

display(readiness)


In [ ]:

# ===== 9. 最终交接包摘要 =====
summary={
    "manual_required_days":len(UNRESOLVED_DATES),
    "manual_completed_days":int(manual_status["saved"].sum()),
    "final_track_count":len(list(FINAL_TRACK.glob("*.npz"))),
    "final_dual_branch_days":len(D),
    "source_counts":meta.source.value_counts().to_dict(),
    "dataset":str(dataset_path),
    "r2_readiness":readiness.to_dict(orient="records"),
}

with open(
    FINAL_REPORT/"R1_FINAL_MANUAL22_HANDOFF.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)

print(json.dumps(summary,indent=2,ensure_ascii=False))
